# 02. Data Structures and Comprehensions

This module explores Python's core collection types and how to choose between them. It covers list operations, hash-based lookups, set algebra, and comprehensions.

The final section connects vanilla Python data structures with NumPy arrays and Pandas DataFrames.

## 1. Lists and Tuples

Lists are **ordered and mutable**. Tuples are **ordered and immutable**. Both store multiple values, but they serve different purposes.

Understanding the time complexity of common operations helps you choose the right structure as your data grows.

### 1.1 Slicing and extended unpacking

Slicing uses `sequence[start:stop:step]` to extract part of a sequence. Extended unpacking uses `*` to collect remaining elements into a list.

In [1]:
def process_signal(readings: list[float]) -> tuple[float, list[float], float]:
    """Extract the first, middle, and last readings."""
    first, *middle, last = readings

    # Select every second reading.
    even_samples = readings[::2]

    # Add values to the end of the new list.
    even_samples.append(99.0)
    even_samples.extend([101.4, 101.3])

    return first, middle, last


signal = [2.3, 3.4, 5.6, 7.4, 4.3, 4.6, 9.5, 9.0]
process_signal(signal)

(2.3, [3.4, 5.6, 7.4, 4.3, 4.6, 9.5], 9.0)

### 1.2 Common list methods and their costs

| Operation | Typical time complexity | Description |
|---|---:|---|
| `append(x)` | O(1) amortized | Add an item to the end |
| `pop()` | O(1) | Remove and return the last item |
| `pop(0)` | O(n) | Remove the first item and shift the remaining items |
| `extend(iterable)` | O(k) | Add `k` items from another iterable |
| `insert(i, x)` | O(n) | Insert an item at a position |

**Performance tip:** Avoid repeatedly inserting at or removing from the beginning of a large list. For a fast FIFO queue, use `collections.deque`.

In [2]:
from collections import deque

queue = deque(["task-1", "task-2", "task-3"])
next_task = queue.popleft()

print("Next task:", next_task)
print("Remaining queue:", queue)

Next task: task-1
Remaining queue: deque(['task-2', 'task-3'])


## 2. Dictionaries and Sets

Dictionaries and sets use hash tables internally. Their membership checks and lookups are **O(1) on average**, while checking whether an item exists in a list takes O(n) time.

These average-case costs are useful when working with large collections.

### 2.1 Dictionaries

A dictionary maps keys to values. Dictionary keys must be hashable.

- `get(key, default)` returns a default value if the key is missing.
- `items()` provides key-value pairs for iteration.
- `setdefault(key, default)` inserts a default only when the key is absent, then returns the value.

In [3]:
stock_counts: dict[str, int] = {
    "VALVE-01": 15,
    "PIPE-02": 42,
    "PUMP-05": 8,
}

# Safe lookup: no KeyError if the key does not exist.
valve_quantity = stock_counts.get("VALVE-01", 0)
missing_quantity = stock_counts.get("MOTOR-99", 0)

print("VALVE-01:", valve_quantity)
print("MOTOR-99:", missing_quantity)

# Iterate over key-value pairs.
for part, quantity in stock_counts.items():
    print(f"{part}: {quantity}")

# Add a default entry only if the key is missing.
stock_counts.setdefault("BOLT-09", 0)
print(stock_counts)

VALVE-01: 15
MOTOR-99: 0
VALVE-01: 15
PIPE-02: 42
PUMP-05: 8
{'VALVE-01': 15, 'PIPE-02': 42, 'PUMP-05': 8, 'BOLT-09': 0}


### 2.2 Sets and set operations

A set is an unordered collection of unique, hashable elements. Converting a list to a set removes duplicate values.

Set algebra is useful for comparing inventories or identifying shared and missing items.

In [4]:
def analyze_inventory(
    plant_a: list[str],
    plant_b: list[str],
    stock_counts: dict[str, int],
) -> dict[str, set[str]]:
    """Compare two inventories and return their set relationships."""
    valve_quantity = stock_counts.get("VALVE-01", 0)

    parts_a = set(plant_a)
    parts_b = set(plant_b)

    return {
        "all_parts": parts_a | parts_b,
        "common_parts": parts_a & parts_b,
        "only_in_a": parts_a - parts_b,
        "only_in_b": parts_b - parts_a,
    }


plant_north = ["VALVE-01", "PIPE-02", "PUMP-05", "VALVE-01", "BOLT-09"]
plant_south = ["PIPE-02", "NUT-03", "VALVE-04", "BOLT-09"]

inventory = {
    "VALVE-01": 15,
    "PIPE-02": 42,
    "PUMP-05": 8,
    "NUT-03": 100,
    "VALVE-04": 23,
    "BOLT-09": 65,
}

results = analyze_inventory(plant_north, plant_south, inventory)
results

{'all_parts': {'BOLT-09',
  'NUT-03',
  'PIPE-02',
  'PUMP-05',
  'VALVE-01',
  'VALVE-04'},
 'common_parts': {'BOLT-09', 'PIPE-02'},
 'only_in_a': {'PUMP-05', 'VALVE-01'},
 'only_in_b': {'NUT-03', 'VALVE-04'}}

The main set operators are:

| Operator | Method | Meaning |
|---|---|---|
| `A / B` | `A.union(B)` | Union: elements in A or B | Note: Is |
| `A & B` | `A.intersection(B)` | Intersection: elements in both |
| `A - B` | `A.difference(B)` | Difference: elements in A but not B |

**Performance tip:** If you repeatedly check membership in a large list, convert it to a set first when appropriate. Average membership lookup can then improve from O(n) to O(1).

In [5]:
large_collection = ["A", "B", "C", "A", "D"]
collection_set = set(large_collection)

print("List membership:", "D" in large_collection)
print("Set membership:", "D" in collection_set)

List membership: True
Set membership: True


## 3. Comprehensions

Comprehensions provide a concise way to build collections from an iterable, optionally filtering its elements.

**Basic syntax:**

```python
[expression for item in iterable if condition]
```

You can also use comprehensions to create dictionaries and sets. Keep them readable: if the logic becomes complicated or deeply nested, use a regular loop.

### 3.1 List, dictionary, and set comprehensions

The example below processes sensor readings:

1. A list comprehension filters valid temperatures and converts them to Kelvin.
2. A dictionary comprehension maps active sensor IDs to their original Celsius readings.
3. A set comprehension extracts unique sensor families.

In [6]:
def build_telemetry_structures(
    sensor_ids: list[str],
    raw_temps: list[float],
) -> tuple[list[float], dict[str, float], set[str]]:
    """Build three collection types from sensor telemetry."""

    # Keep positive readings and convert them to Kelvin.
    valid_kelvin = [
        round(temp + 273.15, 2)
        for temp in raw_temps
        if temp > 0
    ]

    # Map each active sensor ID to its Celsius reading.
    sensor_map = {
        sensor_id: temp
        for sensor_id, temp in zip(sensor_ids, raw_temps, strict=True)
        if temp > 0
    }

    # Extract unique sensor families, such as TEMP and ENV.
    sensor_families = {
        sensor_id.split("-")[0]
        for sensor_id in sensor_ids
    }

    return valid_kelvin, sensor_map, sensor_families

### 3.2 Run the telemetry example

The sample data includes valid readings and values representing faulty or inactive sensors. The function filters out readings that are zero or negative.

In [6]:
sensor_ids = [
    "TEMP-001", "TEMP-002", "ENV-001", "ENV-002",
    "PROC-001", "PROC-002", "TEMP-003", "ENV-003",
]

raw_temps = [
    22.5, -99.0, 18.1, 0.0,
    45.8, 52.3, 24.7, -5.0,
]

kelvin_values, active_sensor_map, sensor_families = build_telemetry_structures(
    sensor_ids=sensor_ids,
    raw_temps=raw_temps,
)

print("Valid readings (Kelvin):")
print(kelvin_values)

print("\nActive sensor map (Celsius):")
print(active_sensor_map)

print("\nUnique sensor families:")
print(sensor_families)

Valid readings (Kelvin):
[295.65, 291.25, 318.95, 325.45, 297.85]

Active sensor map (Celsius):
{'TEMP-001': 22.5, 'ENV-001': 18.1, 'PROC-001': 45.8, 'PROC-002': 52.3, 'TEMP-003': 24.7}

Unique sensor families:
{'TEMP', 'ENV', 'PROC'}


**Readability rule:** A comprehension should make the transformation easy to understand. If it needs several nested loops or complicated conditions, a traditional `for` loop is usually clearer.

## 4. Vanilla Python vs. NumPy and Pandas

Understanding Python collections makes it easier to understand the data structures used by scientific libraries.

| Vanilla Python | NumPy / Pandas equivalent | Typical use |
|---|---|---|
| `list[float]` | 1D `np.ndarray` / `pd.Series` | Sensor readings and time series |
| `list[list[float]]` | 2D `np.ndarray` | Matrices and linear algebra |
| `list[dict]` | `pd.DataFrame` | Tabular records and production logs |

The following example filters records in vanilla Python and then performs the same task with Pandas.

### 4.1 Vanilla Python: a list of dictionaries

A list of dictionaries is a common structure for records loaded from JSON files or APIs.

In [7]:
records = [
    {"motor": "M1", "rpm": 1200.0, "temp": 65.0},
    {"motor": "M2", "rpm": 1450.0, "temp": 78.5},
]

hot_motors_vanilla = [
    row["motor"]
    for row in records
    if row["temp"] > 70.0
]

hot_motors_vanilla

['M2']

### 4.2 Pandas: a DataFrame

Pandas can construct a DataFrame directly from a list of dictionaries. Boolean filtering then selects the records that meet the temperature condition.

In [9]:
import pandas as pd

df = pd.DataFrame(records)

hot_motors_pandas = df.loc[df["temp"] > 70.0, "motor"].tolist()

print(df)
print("\nHot motors:", hot_motors_pandas)

  motor     rpm  temp
0    M1  1200.0  65.0
1    M2  1450.0  78.5

Hot motors: ['M2']


## Conclusion

This module covered the main Python collection types and when to use them:

- **Lists and tuples** store ordered sequences.
- **Dictionaries** map keys to values.
- **Sets** store unique elements and support fast average-case membership checks.
- **Comprehensions** build collections concisely when the logic remains readable.

These structures are the foundation for working with NumPy arrays and Pandas DataFrames. Choosing the right collection—and understanding its time complexity—helps make programs clearer and more efficient.